# connections-rl — entropy + KL-from-reference per checkpoint (7B)

Two analyses in one pass over the GRPO checkpoints, because both are read off
the same forward passes:

1. **Policy entropy per checkpoint** — does the semantic collapse in the
   step 100→150 window coincide with an entropy-collapse event?
2. **KL(policy ‖ SFT init) per checkpoint**, plotted against the semantic
   score. This is the Gao/Schulman/Hilton over-optimization x-axis; without it
   "over-optimization" is a loose word rather than a technical claim.
   KL from the raw base model is recorded too.

Method: one 4-bit base model in memory with every checkpoint attached as a
named PEFT adapter, so switching checkpoints costs no reload. Sampling at the
training temperature (0.9) on the **val** split, so the test set is never reused.

Settings → GPU T4 x2 (only 1 used), Internet on. Save & Run All, walk away.
~60-90 min for 10 checkpoints × 100 puzzles.

In [ ]:
# Cell 1 — setup: repo, data, SFT adapter, GRPO checkpoints
import os, re, pathlib
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
HF_USER = 'jacksonlukas'

!git clone https://github.com/jacksonmlukas/connections-rl.git
%cd connections-rl
!pip install -q -e ".[train]" bitsandbytes
!pip uninstall -q -y torchao
!git clone --depth 1 https://github.com/jacksonmlukas/gvc-local.git /kaggle/working/gvc-local
os.environ['CONNECTIONS_PUZZLES'] = '/kaggle/working/gvc-local/data/puzzles/tagged_connections.json'
!python -m connections_rl.data.build --out data/splits

from huggingface_hub import snapshot_download
snapshot_download(f'{HF_USER}/connections-rl-sft-7b', local_dir='adapters/sft-7b',
                  token=os.environ['HF_TOKEN'])
snapshot_download(f'{HF_USER}/connections-rl-grpo-7b-ckpt', local_dir='ckpts-7b',
                  token=os.environ['HF_TOKEN'], allow_patterns=['checkpoint-*/*'])
steps = sorted(int(re.search(r'\d+', d.name).group())
               for d in pathlib.Path('ckpts-7b').glob('checkpoint-*'))
print('checkpoints:', steps)

In [ ]:
# Cell 2 — entropy + KL sweep (base, SFT init, then every GRPO checkpoint)
specs = ['base=none', 'sft=adapters/sft-7b'] + \
        [f'ckpt-{s}=ckpts-7b/checkpoint-{s}' for s in steps]
print(' '.join(specs))

import subprocess
subprocess.run(['python', '-m', 'connections_rl.eval.entropy_kl',
                '--model', 'Qwen/Qwen2.5-7B-Instruct', '--load-in-4bit',
                '--sft-adapter', 'adapters/sft-7b',
                '--checkpoints', *specs,
                '--puzzles', 'data/splits/puzzles_val.json',
                '--n', '100', '--temperature', '0.9', '--max-new-tokens', '256',
                '--out', 'results-analysis/entropy-kl-7b'], check=True)

In [ ]:
# Cell 3 — read out the phase transition
import json
pts = json.load(open('results-analysis/entropy-kl-7b.json'))
print(f"{'step':>5} {'entropy':>9} {'KL|sft/seq':>11} {'KL|base/tok':>12} {'struct':>8} {'semantic':>9}")
for p in pts:
    print(f"{p['step']:>5} {p['entropy_per_token']:>9.4f} {p['kl_from_sft_per_sequence']:>11.2f} "
          f"{p['kl_from_base_per_token']:>12.4f} {p['structural_valid_rate']:>8.3f} "
          f"{p['semantic_groups_correct']:>9.3f}")

grpo = [p for p in pts if p['name'].startswith('ckpt')]
if len(grpo) > 1:
    e0, e1 = grpo[0]['entropy_per_token'], grpo[-1]['entropy_per_token']
    print(f"\nentropy {e0:.4f} -> {e1:.4f}  ({e1/e0 if e0 else float('nan'):.3f}x)")
    drops = [(grpo[i]['step'], grpo[i-1]['entropy_per_token'] - grpo[i]['entropy_per_token'])
             for i in range(1, len(grpo))]
    step, d = max(drops, key=lambda t: t[1])
    print(f"largest single-interval entropy drop: {d:.4f} nats, ending at step {step}")

In [ ]:
# Cell 4 — figure + durable copy to the Hub
from IPython.display import Image, FileLink, display
display(Image('results-analysis/entropy-kl-7b.png'))

from huggingface_hub import HfApi
api = HfApi()
repo = f'{HF_USER}/connections-rl-results'
api.create_repo(repo, repo_type='dataset', exist_ok=True)
api.upload_folder(folder_path='results-analysis', repo_id=repo, repo_type='dataset',
                  path_in_repo='results-analysis')
print(f'pushed -> huggingface.co/datasets/{repo}/tree/main/results-analysis')
display(FileLink('results-analysis/entropy-kl-7b.json'))